# LangGraph StateGraph 기초: 레스토랑 메뉴 챗봇

## 이 노트북의 역할
LangGraph의 핵심 구성요소인 **State(상태) · Node(노드) · Edge(엣지)** 를 레스토랑 메뉴 추천 예제로 익힙니다. 두 단계로 진행합니다.

| 단계 | 내용 | 배우는 것 |
|---|---|---|
| 2. StateGraph | LLM 없이 함수만으로 `선호도 생성 → 메뉴 추천 → 메뉴 정보` 를 순서대로 실행 | State 정의, Node 함수, `add_edge`, `compile`, `invoke` |
| 3. 조건부 엣지 + RAG | 질문이 메뉴 관련이면 벡터DB 검색 후 답변, 아니면 일반 답변 | `add_conditional_edges`, 라우팅 함수, RAG 노드 |

## 전체 흐름 (3단계 그래프)
```text
START → get_user_query → analyze_input ─┬─ (메뉴 관련)  → search_menu_info → generate_menu_response ──┐
                         (LLM 으로 판정)  │                                                              ├→ END
                                         └─ (메뉴 무관)  → generate_general_response ──────────────────┘
```

| 노드 | 하는 일 | 상태 변경 |
|---|---|---|
| `get_user_query` | 입력 질문을 상태에 담음 | `user_query` |
| `analyze_input` | LLM 이 메뉴 관련 질문인지 `True/False` 판정 | `is_menu_related` |
| `search_menu_info` | 메뉴 벡터DB(FAISS)에서 유사 문서 검색 | `search_results` |
| `generate_menu_response` | 검색 결과를 근거로 LLM 이 답변 생성 | `final_answer` |
| `generate_general_response` | 메뉴와 무관한 질문에 LLM 이 일반 답변 | `final_answer` |

> 분기는 노드가 아니라 **엣지의 조건**입니다. 라우팅 함수 `decide_next_step` 이 `is_menu_related` 를 읽어 다음 노드 이름을 반환합니다.

## 사용 모델 (`get_llm()`)
- `PROVIDER` 값(`"groq"` / `"openai"` / `"upstage"`, 기본값 `"upstage"`)만 바꾸면 이후 모든 셀의 LLM 이 바뀝니다. 바꾼 뒤에는 LLM 설정 셀부터 다시 실행하세요.
- **임베딩은 항상 Upstage** (`solar-embedding-1-large`) 입니다. `../db/menu_db` 를 이 모델로 만들기 때문에 다른 임베딩으로 바꾸면 검색이 되지 않습니다. 따라서 어떤 제공자를 골라도 `UPSTAGE_API_KEY` 는 필요합니다.
- **메뉴 벡터DB 생성**: 3단계에서 `../data/restaurant_menu.txt` 로 `../db/menu_db` 를 만들어 저장합니다. 이미 있으면 건너뜁니다(`REBUILD_MENU_DB = False`). `04_ReAct_Memory`, `05_AgenticRAG` 노트북이 이 저장소를 사용하므로 **이 노트북을 먼저 실행**하세요.
- LLM·임베딩 호출은 사용 요금이 발생할 수 있습니다.

## 실행 시 주의
- `compile()` 이후에 `add_edge` 셀을 다시 실행하면 `Adding an edge to a graph that has already been compiled` 경고가 납니다. 그래프 구성 셀(`StateGraph(...)` 부터)을 **처음부터 순서대로** 다시 실행하세요.
- 2단계(`MenuRecommendState`)와 3단계(`MenuState`)는 서로 다른 상태 스키마입니다.
- 참고: 도구를 LLM 이 자율 호출하는 에이전트는 `create_agent`, 이 노트북처럼 경로를 개발자가 고정하는 워크플로우는 `StateGraph` 가 적합합니다.

## 1. 환경 설정

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 제공자별 API 키 (키 값은 출력하지 않음) — 사용할 제공자의 키만 있으면 되지만,
# 벡터DB 임베딩은 항상 Upstage 를 쓰므로 UPSTAGE_API_KEY 는 필수
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

`(2) 기본 라이브러리`

In [ ]:
import re
import os, json

from textwrap import dedent
from pprint import pprint

# 학습용 노트북이므로 경고 출력을 숨김
import warnings
warnings.filterwarnings("ignore")

`(3) LLM 설정 (Groq / OpenAI / Upstage 선택)`
- `PROVIDER` 값만 바꾸면 이후 모든 셀의 모델이 바뀝니다. (Groq 는 OpenAI 호환 API 라서 `ChatOpenAI` 에 `base_url` 만 다르게 연결)
- 바꾼 뒤에는 이 셀부터 다시 실행하세요. OpenAI 선택 시 사용 요금이 발생합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage

# 사용할 LLM 제공자 선택: "groq", "openai", "upstage" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "upstage"

# 제공자별 설정 (cls: 사용할 LangChain 채팅 모델 클래스)
LLM_CONFIGS = {
    "groq": {
        "cls": ChatOpenAI,  # Groq 는 OpenAI 호환 API 라서 base_url 만 다름
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "cls": ChatOpenAI,
        "api_key": OPENAI_API_KEY,
        "base_url": None,  # 생략 시 OpenAI 기본 엔드포인트 사용
        "model": "gpt-4o-mini",  # 테스트에는 작은 모델 사용 (대안: "gpt-4o")
    },
    "upstage": {
        "cls": ChatUpstage,
        "api_key": UPSTAGE_API_KEY,
        "base_url": "https://api.upstage.ai/v1",
        "model": "solar-pro3",
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, provider: str | None = None, verbose: bool = True, **kwargs):
    """선택한 제공자(기본값: PROVIDER)의 채팅 모델 인스턴스를 생성합니다.
    verbose=True 이면 생성된 모델 정보를 출력합니다.
    kwargs: 모델 클래스에 그대로 전달할 추가 옵션 (예: seed=42, timeout=60)"""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = dict(LLM_CONFIGS[provider])
    model_cls = config.pop("cls")
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = model_cls(**config, temperature=temperature, max_retries=MAX_RETRIES, **kwargs)
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

## 2. StateGraph
- 상태(state)를 기반으로 작동하는 그래프 구조
- 실습: 레스토랑 메뉴 추천 시스템
    - 사용자의 선호도에 따라 메뉴를 추천하고, 메뉴에 대한 정보를 제공

`(1) 상태(State)`
- 상태는 그래프가 처리하는 데이터의 구조를 정의
- 기존 상태를 override (덮어쓰기)

In [ ]:
from typing import TypedDict

# 상태 Schema 정의 - 사용자의 선호도, 추천된 메뉴, 메뉴 정보를 저장
class MenuRecommendState(TypedDict):
    """2단계 그래프의 공유 상태. 노드는 바꾸고 싶은 키만 반환하면 해당 키만 덮어쓴다."""
    user_preference: str
    recommended_menu: str
    menu_info: str

`(2) 노드(Node)`
- 노드는 그래프에서 실제 작업을 수행하는 함수

In [ ]:
import random

def get_user_preference(state: MenuRecommendState) -> MenuRecommendState:
    """[노드 1] 선호도 후보 중 하나를 무작위로 골라 user_preference 에 저장한다."""
    print("---랜덤 사용자 선호도 생성---")
    preferences = ["육류", "해산물", "채식", "아무거나"]
    preference = random.choice(preferences)
    print(f"생성된 선호도: {preference}")
    return {"user_preference": preference}

def recommend_menu(state: MenuRecommendState) -> MenuRecommendState:
    """[노드 2] 선호도에 맞는 메뉴를 골라 recommended_menu 에 저장한다."""
    print("---메뉴 추천---")
    preference = state['user_preference']
    if preference == "육류":
        menu = "스테이크"
    elif preference == "해산물":
        menu = "랍스터 파스타"
    elif preference == "채식":
        menu = "그린 샐러드"
    else:
        menu = "오늘의 쉐프 특선"
    print(f"추천 메뉴: {menu}")
    return {"recommended_menu": menu}

def provide_menu_info(state: MenuRecommendState) -> MenuRecommendState:
    """[노드 3] 추천 메뉴의 설명과 가격을 menu_info 에 저장한다."""
    print("---메뉴 정보 제공---")
    menu = state['recommended_menu']
    if menu == "스테이크":
        info = "최상급 소고기로 만든 juicy한 스테이크입니다. 가격: 30,000원"
    elif menu == "랍스터 파스타":
        info = "신선한 랍스터와 al dente 파스타의 조화. 가격: 28,000원"
    elif menu == "그린 샐러드":
        info = "신선한 유기농 채소로 만든 건강한 샐러드. 가격: 15,000원"
    else:
        info = "쉐프가 그날그날 엄선한 특별 요리입니다. 가격: 35,000원"
    print(f"메뉴 정보: {info}")
    return {"menu_info": info}

`(3) 그래프(Graph) 구성`
- 정의한 구성 요소들을 사용하여 전체 그래프를 빌드

In [ ]:
from langgraph.graph import StateGraph, START, END

# 그래프 빌더 생성
builder = StateGraph(MenuRecommendState)

# 노드 추가: add_node(노드 이름, 실행할 함수)
builder.add_node("get_preference", get_user_preference)
builder.add_node("recommend", recommend_menu)
builder.add_node("provide_info", provide_menu_info)

# 엣지 추가: 실행 순서를 정의 (START → get_preference → recommend → provide_info → END)
builder.add_edge(START, "get_preference")
builder.add_edge("get_preference", "recommend")
builder.add_edge("recommend", "provide_info")
builder.add_edge("provide_info", END)

# 그래프 컴파일: 검증 후 invoke() 로 실행 가능한 객체를 반환
graph = builder.compile()
graph

In [ ]:
from IPython.display import Image, display

# 그래프 시각화
display(Image(graph.get_graph().draw_mermaid_png()))

mermaid_code = graph.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

* https://mermaid.live/ 에서  mermain_code 로 직접 확인할 수도 있습니다.

* [Graph이미지](https://www.mermaidchart.com/play?utm_source=mermaid_live_editor&utm_medium=share#pako:eNplkEEOwiAQRa9C6qZNrFYXLmjTlUdwp6ZBOtOSIBCKJsZ4d1tqiOiKGebBf_BMuG4hoUlnmenJYV-e1Mk1zeCYHZf0WJk6dNXa1OeMUorCDm4CO3CNsYBgQXFI4zabCAtcX6-g2jRUft9YfRctNEKhTr-bbBYYuRDv6xAu2ZwdtEie1ySOLv_lPBUcysjNz74lyl9FT3xM_JCPGsMekLSA7CYdQSElXeAWC8SlFAryHkTXO7pZbaMD_vM8nmvDuHAPWkTA9MDPdRe87JAnrzcCGprA)

In [ ]:
# 그래프 실행

def print_result(result: MenuRecommendState):
    """그래프 실행 결과(최종 상태)를 보기 좋게 출력한다."""
    print("\n=== 결과 ===")
    print("선호도:", result['user_preference'])
    print("추천 메뉴:", result['recommended_menu'])
    print("메뉴 정보:", result['menu_info'])
    print("============\n")


# 초기 입력 (노드가 값을 채우므로 빈 값으로 시작)
inputs = {"user_preference": ""}

# 여러 번 실행하여 테스트 (선호도가 무작위라서 실행마다 결과가 다름)
for _ in range(2):
    result = graph.invoke(inputs)
    print(type(result))
    print_result(result)
    print("*"*100)
    print()

## 3. 조건부 엣지(Edge) : RAG 적용
- 엣지는 노드 간의 연결을 정의
- 조건부 엣지: 사용자 입력이 메뉴 관련인지 여부에 따라 다른 경로로 진행

`(1) State 정의`
- 사용자 입력이 메뉴 추천이면 벡터저장소에서 검색하여 RAG Chain을 실행하고,
- 그렇지 않은 경우에는 LLM이 답변을 생성

In [ ]:
from typing import List, TypedDict

# state 스키마 
class MenuState(TypedDict):
    user_query: str
    is_menu_related: bool
    search_results: List[str]
    final_answer: str

`(2) 벡터저장소 생성 및 검색 도구`
* 메뉴판 텍스트(`restaurant_menu.txt`)를 **메뉴 항목 단위**로 나눠 벡터저장소(FAISS)를 만들고 `../db/menu_db` 에 저장합니다. 한 번 만들어 두면 이후에는 저장된 것을 불러다 씁니다.
* 항목 단위로 나누면 "스테이크 가격" 같은 질문에 해당 메뉴 하나가 정확히 검색됩니다.

In [ ]:
import re
from typing import List

from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document


def split_menu_items(document: Document) -> List[Document]:
    """메뉴 텍스트 문서를 '1. 메뉴명 ...' 항목 단위로 나눠, 항목마다 Document 하나로 만듭니다.

    metadata: source(원본 파일 경로), menu_number(순번, 1부터), menu_name(메뉴 이름)
    """
    # '숫자. ' 로 시작해 다음 '\n\n숫자.' 또는 문서 끝 직전까지 (re.DOTALL: '.' 이 줄바꿈도 포함)
    pattern = r"(\d+\.\s.*?)(?=\n\n\d+\.|$)"
    menu_items = re.findall(pattern, document.page_content, re.DOTALL)

    menu_documents = []
    for i, item in enumerate(menu_items, 1):
        # 첫 줄 '1. 시그니처 스테이크' 에서 '숫자.' 를 떼고 메뉴 이름만 추출
        menu_name = item.split("\n")[0].split(".", 1)[1].strip()
        menu_documents.append(Document(
            page_content=item.strip(),
            metadata={"source": document.metadata["source"], "menu_number": i, "menu_name": menu_name},
        ))
    return menu_documents


# 상수: 데이터 · 저장소 경로 (경로는 이 노트북 위치 기준)
MENU_DATA_PATH = "../data/restaurant_menu.txt"
EMBEDDING_MODEL = "solar-embedding-1-large"  # 저장소를 만들 때와 검색할 때 같은 모델이어야 함
MENU_DB_PATH = "../db/menu_db"
REBUILD_MENU_DB = False  # True 이면 이미 있는 저장소도 다시 만듦 (임베딩 요금 발생)

# 메뉴판 텍스트를 로드해 항목별로 분할 (임베딩 호출 없음)
documents = TextLoader(MENU_DATA_PATH, encoding="utf-8").load()
menu_documents = []
for doc in documents:
    menu_documents += split_menu_items(doc)

print(f"총 {len(menu_documents)}개의 메뉴 항목")
print(menu_documents[0].page_content[:120], "...")
print(menu_documents[0].metadata)

In [ ]:
import os

from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings

embeddings_model = UpstageEmbeddings(model=EMBEDDING_MODEL)

# 벡터저장소 생성·저장: 이미 있으면 건너뜀 (생성 시 임베딩 API 호출 → 사용 요금 발생)
if os.path.exists(MENU_DB_PATH) and not REBUILD_MENU_DB:
    print(f"[건너뜀] 이미 있음: {MENU_DB_PATH}")
else:
    os.makedirs(os.path.dirname(MENU_DB_PATH), exist_ok=True)
    FAISS.from_documents(documents=menu_documents, embedding=embeddings_model).save_local(MENU_DB_PATH)
    print(f"[저장] {len(menu_documents)}개 항목 → {MENU_DB_PATH}")

* 저장된 벡터저장소를 불러와 검색 도구로 사용합니다.

In [ ]:
# 상수: 검색해 올 유사 문서 수 (임베딩 모델·저장소 경로는 위에서 정의)
SEARCH_K = 2

# menu db 벡터 저장소 로드
menu_db = FAISS.load_local(
    MENU_DB_PATH,
    embeddings_model, 
    allow_dangerous_deserialization=True  # 직접 만든 로컬 저장소만 로드할 것 (pickle 역직렬화 허용)
)

print(f" 저장된 벡터 개수: {menu_db.index.ntotal}")
print(f" 벡터 차원: {menu_db.index.d}")
print(f" 인덱스 타입: {type(menu_db.index)}")

`(3) 노드(Node)`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 위 (3) LLM 설정의 get_llm() 으로 모델 생성 — 제공자 전환은 PROVIDER 또는 get_llm(provider="openai")
llm = get_llm(temperature=0.5)

In [ ]:

def get_user_query(state: MenuState) -> MenuState:
    """[노드 1] 입력 질문을 상태에 담는다. (대화형으로 바꾸려면 input() 으로 받으면 됨)"""
    print('1 ==> get_user_query()')
    #user_query = input("무엇을 도와드릴까요? ")
    user_query = state['user_query']
    return {"user_query": user_query}


In [ ]:

def analyze_input(state: MenuState) -> MenuState:
    """[노드 2] LLM 으로 질문이 메뉴 관련인지 판정해 is_menu_related 에 저장한다."""
    print('2 ==> analyze_input()')
    analyze_template = """
    사용자의 입력을 분석하여 레스토랑 메뉴 추천이나 음식 정보에 관한 질문인지 판단하세요.

    사용자 입력: {user_query}

    레스토랑 메뉴나 음식 정보에 관한 질문이면 "True", 아니면 "False"로 답변하세요.
    다른 설명 없이 True 또는 False 한 단어만 출력하세요.

    답변:
    """
    analyze_prompt = ChatPromptTemplate.from_template(analyze_template)
    analyze_chain = analyze_prompt | llm | StrOutputParser()
    
    result = analyze_chain.invoke({"user_query": state['user_query']})
    print('====> result.strip().lower()', result.strip().lower())
    # "True." 나 "**True**" 처럼 기호가 붙어도 처리하도록, 처음 나오는 true/false 단어로 판정
    # (판정 불가이면 False → 일반 답변 경로)
    match = re.search(r"true|false", result.lower())
    is_menu_related = bool(match) and match.group() == "true"
    
    return {"is_menu_related": is_menu_related}

In [ ]:

# 'search_menu_info' 함수는 벡터 데이터베이스에서 관련 메뉴 정보를 검색합니다.
def search_menu_info(state: MenuState) -> MenuState:
    """[노드 3] 질의와 유사한 메뉴 문서를 검색해 search_results 에 저장한다."""
    print('3 ==> search_menu_info()')
    
    # 'menu_db'에서 사용자 질의와 유사한 문서 SEARCH_K 개를 검색합니다.
    results = menu_db.similarity_search(state['user_query'], k=SEARCH_K)
    
    # 검색된 문서들의 내용만 추출하여 리스트로 만듭니다.
    search_results = [doc.page_content for doc in results]
    
    # 검색 결과를 상태에 저장하여 다음 단계에 전달합니다.
    return {"search_results": search_results}

# 'generate_menu_response' 함수는 검색된 메뉴 정보를 활용하여 답변을 생성합니다.
def generate_menu_response(state: MenuState) -> MenuState:
    """[노드 4] 검색 결과를 근거로 LLM 이 메뉴 답변을 생성해 final_answer 에 저장한다."""
    print('4 ==> generate_menu_response()')
    
    # 답변 생성에 사용될 프롬프트 템플릿을 정의합니다.
    response_template = """
    사용자 입력: {user_query}
    메뉴 관련 검색 결과: {search_results}

    위 정보를 바탕으로 사용자의 메뉴 관련 질문에 대한 상세한 답변을 생성하세요. 
    검색 결과의 정보를 활용하여 정확하고 유용한 정보를 제공하세요.

    답변:
    """
    
    # 템플릿을 기반으로 프롬프트 체인을 구성합니다.
    # 'response_prompt'가 LLM에 입력되고, 결과는 문자열로 파싱됩니다.
    response_prompt = ChatPromptTemplate.from_template(response_template)
    response_chain = response_prompt | llm | StrOutputParser()
    
    # 구성된 체인을 실행하여 최종 답변을 얻습니다.
    final_answer = response_chain.invoke({"user_query": state['user_query'], "search_results": state['search_results']})
    print(f"\n메뉴 어시스턴트: {final_answer}")
    
    # 생성된 답변을 상태에 저장합니다.
    return {"final_answer": final_answer}

# 'generate_general_response' 함수는 메뉴와 관련 없는 일반적인 질문에 응답합니다.
def generate_general_response(state: MenuState) -> MenuState:
    """[노드 5] 메뉴와 무관한 질문에 LLM 이 일반 답변을 생성해 final_answer 에 저장한다."""
    print('5 ==> generate_general_response()')
    
    # 일반적인 대화에 사용될 프롬프트 템플릿을 정의합니다.
    response_template = """
    사용자 입력: {user_query}

    위 입력은 레스토랑 메뉴나 음식과 관련이 없습니다. 
    일반적인 대화 맥락에서 적절한 답변을 생성하세요.

    답변:
    """
    
    # 템플릿을 기반으로 프롬프트 체인을 구성합니다.
    response_prompt = ChatPromptTemplate.from_template(response_template)
    response_chain = response_prompt | llm | StrOutputParser()
    
    # 체인을 실행하여 일반적인 답변을 생성합니다.
    final_answer = response_chain.invoke({"user_query": state['user_query']})
    print(f"\n일반 어시스턴트: {final_answer}")
    
    # 생성된 답변을 상태에 저장합니다.
    return {"final_answer": final_answer}

`(4) 엣지(Edge)`
* 라우팅(분기) 함수는 **노드가 아니라 엣지의 조건**입니다. 상태를 읽고 **다음에 갈 노드 이름**을 반환합니다. (상태는 바꾸지 않음)
* 반환 타입을 `Literal[...]` 로 적어 두면 가능한 경로가 명확해지고, 시각화에도 도움이 됩니다.

In [ ]:
from typing import Literal

def decide_next_step(state: MenuState) -> Literal["search_menu_info", "generate_general_response"]:
    """[라우팅] is_menu_related 값에 따라 다음에 실행할 노드 이름을 반환한다."""
    # 현재 상태에서 'is_menu_related' 값을 출력하여 다음 결정 과정을 로그로 남깁니다.
    print('6 Edge ==> decide_next_step() is_menu_related = ', state['is_menu_related'])
    
    # 만약 'is_menu_related'가 True이면 (질문이 메뉴 관련 질문일 때),
    # 메뉴 정보를 검색하는 노드로 이동하도록 "search_menu_info"를 반환합니다.
    if state['is_menu_related']:
        return "search_menu_info"
    
    # 그렇지 않다면 (질문이 일반적인 질문일 때),
    # 일반적인 답변을 생성하는 노드로 이동하도록 "generate_general_response"를 반환합니다.
    else:
        return "generate_general_response"

`(5) 그래프(Graph) 구성`

In [ ]:
from langgraph.graph import StateGraph, START, END

# StateGraph 클래스를 사용하여 그래프를 구성합니다. 
# MenuState는 그래프가 공유하는 상태(state)의 스키마를 정의합니다.
builder = StateGraph(MenuState)

# --- 노드 추가 ---
# 각 노드는 그래프의 한 단계를 나타내며, 특정 작업을 수행합니다.
# "get_user_query": 사용자의 입력(질문)을 가져오는 노드
# MenuState 객체의 user_query 변수에 값을 저장하는 노드
builder.add_node("get_user_query", get_user_query)

# "analyze_input": 사용자 질의가 메뉴와 관련 있는지 분석하는 노드
# MenuState 객체의 is_menu_related 변수에 값을 저장하는 노드
builder.add_node("analyze_input", analyze_input)

# "search_menu_info": 메뉴 관련 질의일 경우, 벡터DB에서 정보를 검색하는 노드
# MenuState 객체의 search_results 변수에 값을 저장하는 노드
builder.add_node("search_menu_info", search_menu_info)

# "generate_menu_response": 검색된 메뉴 정보를 바탕으로 답변을 생성하는 노드
# MenuState 객체의 final_answer 변수에 값을 저장하는 노드
builder.add_node("generate_menu_response", generate_menu_response)

# "generate_general_response": 메뉴와 관련 없는 일반적인 질문에 답변하는 노드
# MenuState 객체의 final_answer 변수에 값을 저장하는 노드
builder.add_node("generate_general_response", generate_general_response)

builder.nodes

In [ ]:

# --- 엣지(연결) 추가 ---
# 그래프의 시작점을 정의하고, 노드들을 순서대로 연결합니다.
# START에서 "get_user_query" 노드로 시작합니다.
builder.add_edge(START, "get_user_query")

# "get_user_query" 노드에서 "analyze_input" 노드로 이동합니다.
builder.add_edge("get_user_query", "analyze_input")

builder.edges


In [ ]:

# --- 조건부 엣지 추가 ---
# "analyze_input" 노드의 결과에 따라 동적으로 다음 노드를 결정합니다.
# decide_next_step 함수가 반환하는 값에 따라 흐름이 분기됩니다.
builder.add_conditional_edges(
    "analyze_input",            # 현재 노드: "analyze_input"
    decide_next_step,           # 라우팅 함수: 'decide_next_step'
    {
        # 'decide_next_step'이 "search_menu_info"를 반환하면, 해당 노드로 이동
        "search_menu_info": "search_menu_info",
        # 'decide_next_step'이 "generate_general_response"를 반환하면, 해당 노드로 이동
        "generate_general_response": "generate_general_response"
    }
)
# 조건부 엣지는 builder.edges 가 아니라 builder.branches 에 저장됨
print("일반 엣지:", builder.edges)
print("조건부 엣지:", dict(builder.branches))

In [ ]:

# --- 최종 엣지 ---
# 분기된 흐름이 최종 답변을 생성하고 종료되도록 연결합니다.
# "search_menu_info" 노드는 "generate_menu_response" 노드로 이어집니다.
builder.add_edge("search_menu_info", "generate_menu_response")

# 메뉴 관련 답변 생성 후 그래프를 종료합니다.
builder.add_edge("generate_menu_response", END)

# 일반적인 답변 생성 후 그래프를 종료합니다.
builder.add_edge("generate_general_response", END)

print(builder.edges)
# --- 그래프 컴파일 ---
# 정의된 노드와 엣지를 기반으로 실행 가능한 그래프를 최적화합니다.
graph = builder.compile()  # 컴파일 이후에는 add_edge 를 추가해도 반영되지 않음

In [ ]:
# 그래프 시각화
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

mermaid_code = graph.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

[Graph이미지](https://mermaidchart.com/play?utm_source=mermaid_live_editor&utm_medium=share#pako:eNp9U12vojAQ_Stk9gUT8GJRwWJ8uf6EfdplQypMhQQKt7TJqvG_by0uCX7cp85Mz5npOW0vkLcFAoWjZF3p_NwnqUhVlvWKSbO4v7fdbsy2H93uz4xSyivZqxvwiCrTPcrsS6M8udN0dkMwwerTGbNKdFq5k8zu98hkXmYNCm2qvHUfC7NhjkDJFA5liX3Xih7d1-UpYwjqF6THndkgHUUxCrfxKLtmg-rREMf3d85UdPIk2vHnFvVm6jvCow_Js992_ISavLL0fspXViXfWmWJdxOS99fwBMuNUf0euVMgZ7pWDq_qmv7ghAece3Ul0C-xOpaKLuZkQrAPy8L9tmN5pU40mABuV3Bvd-CHNc_BM0-3KoByVvfoQYOyYbccLqlwnBRUiQ2mQE14P04KqbgaXsfEr7ZtgCqpDVO2-lj-T3RXGJ37ipl_0YzNpdGI8rPVQgEly43tAfQCf4FGRksUBmQVbtaLcLlZeXACuljG83hN4g1ZxeEyIvHVg7MdGsyjKCQRIYt1SIIgjKLrP64fR1Y)

`(6) Graph 실행`

In [ ]:
# 메뉴 관련 질문 → search_menu_info → generate_menu_response 경로
query = "레스토랑의 메뉴를 추천해 주세요"
initial_state = {'user_query':query}
graph.invoke(initial_state)

In [ ]:
# 메뉴 관련 질문(해산물) → 검색 결과가 질문에 맞게 달라지는지 확인
query = "레스토랑의 해산물 메뉴를 추천해 주세요"
initial_state = {'user_query':query}
graph.invoke(initial_state)

In [ ]:
# 메뉴와 무관한 질문 → generate_general_response 경로
query = "안녕하세요? 내일 날씨는 어떤가요?"
initial_state = {'user_query':query}
graph.invoke(initial_state)

In [ ]:
# while True:
#     initial_state = {'user_query':''}
#     graph.invoke(initial_state) 
#     continue_chat = input("다른 질문이 있으신가요? (y/n): ").lower()
#     if continue_chat != 'y':
#         print("대화를 종료합니다. 감사합니다!")
#         break